# Reproducing Semenov et al. (2025), Figure S2, from dandiset 001271

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025). Excitation and polarization of
isolated neurons by high-frequency sine waves in the context of temporal interference
stimulation. *Cell Reports Physical Science* 6, 102660,
[doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660).

Dataset. [DANDI:001271](https://dandiarchive.org/dandiset/001271/0.250805.1352), "Tetrodotoxin
Blockade of Sodium Channels Abolishes Action Potential Formation During Intense Sinewave
Stimulation", version 0.250805.1352, streamed from the archive. The paper cites the dandiset
without a version, so the most recent published version is pinned. The earlier published version,
0.250701.1624, holds the same 287 assets with the same blobs (checked below), so the choice does
not change the data.

Claims. `claims/001271/10.1016-j.xcrp.2025.102660.yaml`, written from the NIH author
manuscript (NIHMS2098259, PMC12320893) with its STAR Methods and the supplement (Figures S1 to S4,
Table S1). The dataset holds the Figure S2 experiment: neurons stimulated by 250-ms, 2-kHz sine
waves before and about 5 min after the solution was replaced by one with 1 uM tetrodotoxin (TTX)
or without it (vehicle). The claims file's other claims are on data in other dandisets
(`data: elsewhere`).

| Claim | Data | Here |
|---|---|---|
| figs2-neurons-tested | this dataset | tested |
| figs2-before-firing-then-depolarization | this dataset | tested |
| figs2-depolarization-reversible | this dataset | tested (blocked if no session qualifies) |
| figs2-ttx-abolishes-aps | this dataset | tested |
| figs2-ttx-blocks-sustained-depolarization | this dataset | tested (blocked if no pair qualifies) |
| figs2-vehicle-keeps-response | this dataset | tested |
| ap-amplitude-duration | this dataset | tested |
| fig3-not-heating | calculated from the paper's values | tested |
| fig3c-sustained-firing-2s | DANDI:001030 | tested in `reproduce_fig3c_001030.ipynb` |
| fig1-pure-sine-excites-all, fig1-20khz-needs-twice-the-field, fig2d-modulated-thresholds-higher, fig2-modulated-locks-to-beats, fig2c-pure-more-aps | DANDI:001032 to 001035 | tested in `reproduce_fig1_fig2_001032.ipynb` |
| fig3b-low-conductivity-halves-threshold | DANDI:001030, 001032 to 001035 (by the claims file) | blocked: no such sessions found, in `reproduce_fig1_fig2_001032.ipynb` |
| figs1-di8-same-as-fluovolt | DANDI:001272 | tested in `reproduce_fig1_fig2_001032.ipynb` |
| fig4-pooled-time-constants, fig4-orientation-and-poles, fig6-sine-less-than-pulse, threshold-depolarization-estimate | DANDI:001273, 001274 | blocked (not attempted in this run) |

The readings of the claims file's header (O1 to O5) are run in every combination. The whole
16 x 16 frame as the soma is shown beside them as a variant that does not decide an outcome.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sinewave as sw

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
print("dandiset", sw.DANDISET_ID, "version", sw.DANDISET_VERSION)

## Data check

In [ ]:
meta = sw.dandiset_metadata()
print(meta["name"])
print(meta["description"])
print("related:", [(r["relation"], r["identifier"]) for r in meta.get("relatedResource", [])])
print("assets summary: files", meta["assetsSummary"]["numberOfFiles"], "subjects",
      meta["assetsSummary"]["numberOfSubjects"], "bytes", meta["assetsSummary"]["numberOfBytes"])
assets = sw.list_assets()
older = sw.list_assets(version="0.250701.1624")
same = {(a["path"], a["blob"]) for a in assets} == {(a["path"], a["blob"]) for a in older}
print(f"{len(assets)} assets in {sw.DANDISET_VERSION}, {len(older)} in 0.250701.1624; "
      f"same paths and blobs: {same}")

In [ ]:
t0 = time.time()
sessions, traces = sw.load_all(assets)
sessions = sw.assign_change(sessions)
print(f"streamed and reduced {len(sessions)} files in {time.time() - t0:.0f} s")

In [ ]:
s = sessions
print("Per file: one ImageSeries 'TimeLapseImaging'.")
print("shapes:", s["shape"].astype(str).value_counts().to_dict(), "| rate (frames/s):",
      s["rate"].value_counts().to_dict(), "| unit:", s["unit"].value_counts().to_dict())
print("session_id:", s["session_id"].value_counts().to_dict())
print("subject age:", s["age"].value_counts().to_dict(), "species:", s["species"].value_counts().to_dict())
print("ImageSeries description equals session description in every file:",
      bool((s["description"] == s["series_description"]).all()))
print("stimulus frequency (kHz) parsed from the descriptions:", s["khz"].value_counts().to_dict())
print("V/cm per V stated in the files:", s["vcm_per_volt_file"].value_counts().to_dict())
print("solution:", s["solution"].value_counts().to_dict(), "| sham sessions:", int(s["sham"].sum()))
print()
print("Subjects (one neuron each, see the identity check below), with the solution change placed as in sinewave.assign_change:")
tab = (s.groupby(["subject", "group"])
         .apply(lambda g: pd.Series(dict(
             sessions=len(g), sham=int(g["sham"].sum()),
             before=int(((g["phase"] == "before") & ~g["sham"]).sum()),
             after=int(((g["phase"] == "after") & ~g["sham"]).sum()),
             volts_before=" ".join(f"{v:g}" for v in sorted(set(g.loc[(g["phase"] == "before") & ~g["sham"], "volt"]))),
             volts_after=" ".join(f"{v:g}" for v in sorted(set(g.loc[(g["phase"] == "after") & ~g["sham"], "volt"]))),
             gap_at_change_min=round(g.loc[g["phase"] == "after", "gap_min"].iloc[0], 1) if (g["phase"] == "after").any() else np.nan,
             largest_gap_min=round(g["gap_min"].max(), 1),
             first=g["start"].min().strftime("%Y-%m-%d %H:%M"))))
         .reset_index())
print(tab.to_string(index=False))
print()
print("neurons per group:", tab.groupby("group").size().to_dict())
print("voltage range (V):", s.loc[~s["sham"], "volt"].min(), "to", s.loc[~s["sham"], "volt"].max())
fm = np.array(s["frame_means"].tolist())
print(f"mean counts per frame: frame 0 {np.median(fm[:, 0]):.0f} (median over files), frames 1-1198 "
      f"{np.median(fm[:, 1]):.0f}, frame 1199 {np.median(fm[:, 2]):.0f}, frame 1198 {np.median(fm[:, 3]):.0f}")
print(f"frame 0 / frames 1-1198: {np.min(fm[:, 0] / fm[:, 1]):.2f} to {np.max(fm[:, 0] / fm[:, 1]):.2f}; "
      f"frame 1199 / frame 1198: {np.min(fm[:, 2] / fm[:, 3]):.3f} to {np.max(fm[:, 2] / fm[:, 3]):.3f}")
print("ROI pixels per session (O1 a: otsu, O1 b: otsu_eroded):")
npx = pd.DataFrame(s["n_pixels"].tolist())
print(npx.describe().loc[["min", "25%", "50%", "75%", "max"]].to_string())
print("sessions where the 2-pixel erosion of O1(b) leaves no pixel:", int((npx["otsu_eroded"] == 0).sum()), "of", len(s))

The files hold 16 x 16 frames, where the paper describes 64 x 64 pixels after 4 x 4 binning
(Methods). Every file's first frame is about three times brighter than the rest and its last
frame a few percent brighter than the one before; both are left out of every trace (sinewave.py).
The images below are mean pre-stimulus frames: the soma fills most of the crop, so there is
little background for a threshold to separate the soma from.

In [ ]:
first = s.sort_values("start").groupby("subject").head(1).sort_values("subject")
fig, axes = plt.subplots(2, 10, figsize=(16, 3.6))
for ax, (_, r) in zip(axes.flat, first.iterrows()):
    m = sw.soma_masks(r["mean_pre"])
    ax.imshow(r["mean_pre"], cmap="gray")
    ax.contour(m["otsu"], [0.5], colors="r", linewidths=0.8)
    if m["otsu_eroded"].any():
        ax.contour(m["otsu_eroded"], [0.5], colors="c", linewidths=0.8)
    ax.set_title(f"{r['subject']}\n{r['group']}", fontsize=7)
    ax.axis("off")
fig.suptitle("Mean pre-stimulus frame of each neuron's first session; red O1(a), cyan O1(b)", fontsize=9)
plt.tight_layout()
plt.show()
print("counts in the images shown:", int(min(r["mean_pre"].min() for _, r in first.iterrows())), "to",
      int(max(r["mean_pre"].max() for _, r in first.iterrows())))

### Duplicates and the identity of neurons across sessions

Each subject is taken as one neuron. Within a subject the sessions should show the same cell;
between subjects, different cells. The mean pre-stimulus frames are compared by correlation.

In [ ]:
print("distinct data arrays (SHA-1 of the frames):", s["sha1"].nunique(), "of", len(s))
wf = np.vstack([t["frame"][1:-1] for t in traces])
c = np.corrcoef(wf)
iu = np.triu_indices(len(wf), 1)
print(f"largest correlation of whole-frame traces between two different files: {c[iu].max():.4f}")
mp = np.vstack([m.ravel() for m in s["mean_pre"]])
cm = np.corrcoef(mp)
subj_arr = s["subject"].astype(str).to_numpy(dtype=object)
same_subj = subj_arr[:, None] == subj_arr[None, :]
within = cm[iu][same_subj[iu]]
between = cm[iu][~same_subj[iu]]
print(f"mean-frame correlation within subjects: median {np.median(within):.3f}, min {within.min():.3f} "
      f"({len(within)} pairs)")
print(f"mean-frame correlation between subjects: median {np.median(between):.3f}, max {between.max():.3f} "
      f"({len(between)} pairs)")
nearest_other = []
for i in range(len(s)):
    own = cm[i, same_subj[i] & (np.arange(len(s)) != i)].min()
    other = cm[i, ~same_subj[i]].max()
    nearest_other.append(own > other)
print("sessions whose least similar same-subject session is more similar than any other subject's:",
      int(np.sum(nearest_other)), "of", len(s))

### Stimulus timing and the clock

The files carry no stimulus record. A 2-kHz sine sampled at 3,086.4 frames/s aliases to
1,086.4 Hz, and at high voltages it is visible in the images. Its envelope gives the onset and
offset of the stimulus in sessions where it rises clearly above the pre-stimulus level. The
contrast below compares AP-like peaks before the 50-ms onset with those during the stimulus.

In [ ]:
rows = []
for _, r in s[~s["sham"]].iterrows():
    env = r["alias"]
    on, off = sw.frames(r["rate"])
    base = env[20:on - 20]
    during = env[on + 20:off - 20]
    ratio = during.mean() / base.mean()
    thr = base.mean() + 0.5 * (during.mean() - base.mean())
    above = np.where(env > thr)[0]
    above = above[(above > 20) & (above < len(env) - 20)]
    rows.append(dict(subject=r["subject"], volt=r["volt"], solution=r["solution"], ratio=ratio,
                     onset_ms=above.min() / r["rate"] * 1000 if len(above) else np.nan,
                     offset_ms=above.max() / r["rate"] * 1000 if len(above) else np.nan))
al = pd.DataFrame(rows)
clear = al[al["ratio"] > 1.5]
print(f"sessions with the alias envelope during the stimulus over 1.5 x its pre-stimulus level: {len(clear)} of {len(al)}")
print("their voltages:", sorted(clear["volt"].unique()))
print(f"half-amplitude crossings (10-ms smoothing): onset median {clear['onset_ms'].median():.1f} ms "
      f"(range {clear['onset_ms'].min():.1f} to {clear['onset_ms'].max():.1f}), offset median "
      f"{clear['offset_ms'].median():.1f} ms (range {clear['offset_ms'].min():.1f} to {clear['offset_ms'].max():.1f})")
print("expected from the Methods: onset 50 ms, offset 300 ms")
sham = s[s["sham"]]
print("sham sessions, alias ratio:", [round(float(r["alias"][160:900].mean() / r["alias"][20:130].mean()), 2) for _, r in sham.iterrows()])

ref = sw.measure(s[~s["sham"]], traces, "frame", "all", "5pct", "5sd")
pk_pre, pk_dur = 0, 0
for _, r in ref.iterrows():
    y = np.nan_to_num(r["x"], nan=0.0)
    found = sw.detect_aps(y, r["rate"], 5.0, 0, len(y))
    pk_pre += int(((found["frame"] >= 20) & (found["frame"] < r["onset"])).sum())
    pk_dur += int(((found["frame"] >= r["onset"]) & (found["frame"] < r["offset"])).sum())
pre_ms = (ref["onset"] - 20).mean() / ref["rate"].mean() * 1000
print(f"AP-like peaks (whole frame, 5% dF) per second: before onset {pk_pre / (len(ref) * pre_ms / 1000):.2f}, "
      f"during the stimulus {pk_dur / (len(ref) * 0.25):.2f}  ({pk_pre} and {pk_dur} peaks in {len(ref)} sessions)")

### Field strength

The files say "1V translates to 36 V/cm". The fields marked in Figure S2 (vehicle neuron 45, 63,
72 V/cm; TTX neurons 90, 108, 117 and 54, 72, 81 V/cm) are compared with the voltages of the
dataset's neurons tested both before and after the change, at 36 and at 18 V/cm per V.

In [ ]:
fig_s2 = {"vehicle": [45, 63, 72], "TTX (a)": [90, 108, 117], "TTX (b)": [54, 72, 81]}
both = {}
for subj, g in s[~s["sham"]].groupby("subject"):
    b = set(g.loc[g["phase"] == "before", "volt"])
    a = set(g.loc[g["phase"] == "after", "volt"])
    both[subj] = (b & a, g["group"].iloc[0])
for name, fields in fig_s2.items():
    for k in (36, 18):
        volts = [round(f / k, 3) for f in fields]
        hits = [subj for subj, (v, grp) in both.items() if set(volts) <= v and grp in name]
        print(f"Figure S2 {name}: {fields} V/cm at {k} V/cm per V = {volts} V; neurons tested at all three "
              f"before and after: {hits}")

## Readings

Every combination of the claims file's readings: O1 soma ROI (a `otsu`, b `otsu_eroded`), O2 F0
(a `all`, b `last20`), O3 AP height (a `5pct`, b `5sd`), O4 sustained depolarization level
(a `5sd`, b `half_ap`), and O5 session pairs (a `every`, b `highest`) where a claim uses pairs.
The whole frame (`frame`) is a variant. Sham sessions are left out. Sessions where the O1(b) mask
is empty have no trace under that reading and are dropped from it.

In [ ]:
sw.READINGS["roi"] = ("otsu", "otsu_eroded", "frame")
stim = s[~s["sham"]]
M = sw.all_combinations(stim, traces)
M = M[M["x"].apply(lambda v: np.isfinite(v).any())].copy()
CLAIMS_ROI = ("otsu", "otsu_eroded")
KEYS = ["roi", "f0", "ap", "sdr"]
print("sessions per ROI reading:", M.groupby("roi")["path"].nunique().to_dict())
print("neurons per ROI reading:", M.groupby("roi")["subject"].nunique().to_dict())


def overall(table, col="outcome"):
    """The outcome all claims-file readings give, or the claims file's rule when they differ."""
    t = table[table["roi"].isin(CLAIMS_ROI)]
    vals = sorted(t[col].unique())
    return vals[0] if len(vals) == 1 else "readings differ: " + ", ".join(vals)

## figs2-neurons-tested

> The tetrodotoxin experiment was replicated in several neurons, with vehicle controls; the
> representative traces of Figure S2 come from at least 4 independent replications in different
> cells.

Readings: (a) at least 4 TTX neurons, (b) at least 4 neurons over TTX and vehicle.

In [ ]:
groups = tab.groupby("group")["subject"].apply(list)
n_ttx = len(groups.get("TTX", []))
n_veh = len(groups.get("vehicle", []))
n_nochange = len(groups.get("no change", []))
print("TTX neurons:", n_ttx, groups.get("TTX", []))
print("vehicle neurons with a solution change:", n_veh, groups.get("vehicle", []))
print("vehicle-labelled neurons with no gap over 3 min (no change visible):", n_nochange, groups.get("no change", []))
print("TTX neurons per recording day:", s[s["group"] == "TTX"].groupby(s["start"].dt.date)["subject"].nunique().to_dict())
print(f"reading (a) TTX >= 4: {n_ttx >= 4}; reading (b) TTX + vehicle >= 4: {n_ttx + n_veh >= 4}")
print("distinct data arrays:", s["sha1"].nunique(), "of", len(s), "(no file deposited twice)")

**Result: reproduced.** The dataset holds 6 TTX neurons (two on each of three recording days) and
11 vehicle neurons with a visible solution change, so both readings (at least 4 TTX neurons; at
least 4 neurons in all) hold. Three more neurons are labelled "without tetrodotoxin (vehicle
control)" but show no gap of more than 3 minutes between sessions, so no solution change can be
placed. All 287 data arrays are distinct.

## figs2-before-firing-then-depolarization

> Before the drug, unmodulated 2-kHz sine waves evoked repetitive APs at lower fields and, at
> the most intense fields, a sustained depolarization in which APs merged into small
> oscillations.

Sessions before the change, all 20 neurons (TTX, vehicle, and the three vehicle-labelled
neurons with no visible change, whose sessions are all before any change).

In [ ]:
B = M[M["phase"] == "before"]


def before_claim(g):
    out = []
    for subj, h in g.groupby("subject"):
        top = h[h["volt"] == h["volt"].max()]
        low = h[h["volt"] < h["volt"].max()]
        out.append(dict(subject=subj, group=h["group"].iloc[0], any_ap=(h["n_ap"] > 0).any(),
                        sd_top=top["sustained"].any(),
                        ap_low_no_sd=((low["n_ap"] > 0) & ~low["sustained"]).any()))
    o = pd.DataFrame(out)
    both_ = o["sd_top"] & o["ap_low_no_sd"]
    ok = o["any_ap"].all() and both_.sum() > len(o) / 2
    return pd.Series(dict(neurons=len(o), with_aps=int(o["any_ap"].sum()), sd_at_top=int(o["sd_top"].sum()),
                          ap_lower_no_sd=int(o["ap_low_no_sd"].sum()), both=int(both_.sum()),
                          sessions_with_sd=int(g["sustained"].sum()), sessions=len(g),
                          outcome="reproduced" if ok else "not reproduced"))


r2 = B.groupby(KEYS).apply(before_claim).reset_index()
print(r2.to_string(index=False))
print("\noutcome under the claims file's readings:", overall(r2))
print("variant, neurons without the three with no visible change:")
print(B[B["group"] != "no change"].groupby(KEYS).apply(before_claim).reset_index()
      [KEYS + ["neurons", "with_aps", "both", "outcome"]].to_string(index=False))

What the last 100 ms of the stimulus hold at each neuron's highest field before the change,
under the first reading (otsu/all/5pct/5sd) and the whole-frame variant: the mean dF, the APs in
the window, and the two O4 levels.

In [ ]:
for roi in ("otsu", "frame"):
    for sdr in ("5sd", "half_ap"):
        m = B[(B["roi"] == roi) & (B["f0"] == "all") & (B["ap"] == "5pct") & (B["sdr"] == sdr)]
        top = m.loc[m.groupby("subject")["volt"].idxmax()]
        if sdr == "5sd":
            print(f"\nROI {roi}, F0 all, APs above 5%:")
            print(top[["subject", "group", "volt", "n_ap", "max_rate_hz", "last100_mean", "ap_in_last100",
                       "pre_sd", "sd_dep_level"]].round(2).to_string(index=False))
            print(f"mean dF over the last 100 ms at the highest field: median {top['last100_mean'].median():.2f}%, "
                  f"range {top['last100_mean'].min():.2f} to {top['last100_mean'].max():.2f}; above 0 in "
                  f"{int((top['last100_mean'] > 0).sum())} of {len(top)}; above 2 pre-stimulus SD in "
                  f"{int((top['last100_mean'] > 2 * top['pre_sd']).sum())}")
        print(f"  O4 level ({sdr}): median {top['sd_dep_level'].median():.2f}%, range "
              f"{top['sd_dep_level'].min():.2f} to {top['sd_dep_level'].max():.2f}; sessions at the top field with "
              f"sustained depolarization {int(top['sustained'].sum())}")

AP count and maximum firing rate (F5) against voltage, before the change, under the first
reading (otsu/all/5pct) and the whole-frame variant. Fields in V/cm are 18 x V on the scale of
Figure S2 (see the field check), 36 x V on the files' stated scale.

In [ ]:
for roi in ("otsu", "frame"):
    m = B[(B["roi"] == roi) & (B["f0"] == "all") & (B["ap"] == "5pct") & (B["sdr"] == "5sd")]
    print(f"\nROI {roi}: AP count (max over repeats) per neuron and voltage")
    print(m.pivot_table(index="subject", columns="volt", values="n_ap", aggfunc="max").to_string(na_rep="."))
    print(f"ROI {roi}: maximum firing rate (Hz) per neuron and voltage")
    print(m.pivot_table(index="subject", columns="volt", values="max_rate_hz", aggfunc="max").round(0).to_string(na_rep="."))

fig, axes = plt.subplots(4, 5, figsize=(16, 9), sharex=True)
mm = B[(B["roi"] == "frame") & (B["f0"] == "all") & (B["ap"] == "5pct") & (B["sdr"] == "5sd")]
lo_y, hi_y = -6, 16
outside = 0
for ax, (subj, h) in zip(axes.flat, mm.groupby("subject")):
    h = h.sort_values("volt")
    for k, (_, r) in enumerate(h.drop_duplicates("volt", keep="last").iterrows()):
        t = np.arange(len(r["x"])) / r["rate"] * 1000
        ax.plot(t, r["x"] + 15 * k, lw=0.4)
        ax.text(395, 15 * k, f"{r['volt']:g}", fontsize=6)
    ax.set_title(f"{subj} ({h['group'].iloc[0]})", fontsize=7)
    ax.axvspan(50, 300, color="0.9", zorder=0)
fig.suptitle("Whole-frame dF (%) before the change, offset by 15% per voltage (V at right); grey: stimulus", fontsize=9)
plt.tight_layout()
plt.show()
allx = np.concatenate([np.asarray(v)[1:-1] for v in mm["x"]])
print(f"plotted dF range {np.nanmin(allx):.1f} to {np.nanmax(allx):.1f}%")

**Result: partially reproduced.** Every neuron fires before the change under the ROI reading
O1(a) with the 5% height (20 of 20; 18 or 19 of 20 with the 5-SD height), and APs without
sustained depolarization occur at a lower field in all of them. But under no combination of the
claims file's readings does any neuron show sustained depolarization (O4) at its highest field
(0 of 20, and 0 of 7 neurons under O1(b)), where the criterion needs more than half. At the
highest field the mean dF over the last 100 ms of the stimulus is above zero in all 20 neurons
(median 2.63% with O1(a)), but below both O4 levels (medians 7.29% and 4.93%), and in 9 of the
20 an AP still falls in that window. The whole-frame variant gives 1 of 20. So the firing part
holds, and a depolarization is present, weaker than the claims file's definition of a sustained
one.

## figs2-depolarization-reversible

> The sustained depolarization induced by intense unmodulated sine waves reversed immediately
> when the stimulus ended, so it was not membrane electroporation, which would take seconds or
> longer to repair.

Sessions with sustained depolarization (O4), before the change and in vehicle neurons. Windows
after the stimulus: (a) 10 to 40 ms after its end, (b) the last 40 ms of the recording.

In [ ]:
def post_window(r, w):
    if w == "10-40ms":
        a = r["offset"] + int(round(0.010 * r["rate"]))
        b = r["offset"] + int(round(0.040 * r["rate"]))
    else:
        b = len(r["x"]) - 1
        a = b - int(round(0.040 * r["rate"]))
    return float(np.nanmean(r["x"][a:b]))


rows = []
for key, g in M.groupby(KEYS):
    q = g[g["sustained"] & ((g["phase"] == "before") | (g["group"] == "vehicle"))]
    for w in ("10-40ms", "last40ms"):
        vals = [post_window(r, w) for _, r in q.iterrows()]
        below = sum(v < lv for v, lv in zip(vals, q["sd_dep_level"]))
        rows.append(dict(zip(KEYS, key), window=w, sessions=len(q), below=below,
                         outcome=("none qualify" if len(q) == 0 else
                                  "reproduced" if below == len(q) else "not reproduced")))
r3 = pd.DataFrame(rows)
print(r3.to_string(index=False))
print("\noutcome under the claims file's readings:", overall(r3))

**Result: blocked (other).** No session qualifies: under every combination of the claims file's
readings, no session shows sustained depolarization (O4), so the criterion has nothing to judge.
The whole-frame variant finds 2 or 3 qualifying sessions under its 5-SD readings, and in all of
them dF falls below the O4 level in both windows after the stimulus.

Variant, not judged: at each neuron's highest field before the change, the mean dF over the
last 100 ms of the stimulus and over the two windows after it (first reading and whole frame).
And the reported check: F0 of each session against the session before it of the same neuron.

In [ ]:
for roi in ("otsu", "frame"):
    m = B[(B["roi"] == roi) & (B["f0"] == "all") & (B["ap"] == "5pct") & (B["sdr"] == "5sd")]
    top = m.loc[m.groupby("subject")["volt"].idxmax()].copy()
    top["post_10_40"] = [post_window(r, "10-40ms") for _, r in top.iterrows()]
    top["post_last40"] = [post_window(r, "last40ms") for _, r in top.iterrows()]
    d1 = top["last100_mean"] - top["post_10_40"]
    print(f"ROI {roi}: last 100 ms of stimulus median {top['last100_mean'].median():.2f}%, 10-40 ms after "
          f"median {top['post_10_40'].median():.2f}%, last 40 ms median {top['post_last40'].median():.2f}%; "
          f"drop to 10-40 ms after in {int((d1 > 0).sum())} of {len(top)} neurons (median {d1.median():.2f} points)")
f0rows = []
for subj, g in stim.sort_values("start").groupby("subject"):
    f0 = [np.nanmean(traces[i]["frame"][1:sw.frames(r)[0]]) for i, r in zip(g["trace_index"], g["rate"])]
    f0 = np.array(f0)
    f0rows += list(100 * (f0[1:] / f0[:-1] - 1))
f0rows = np.array(f0rows)
print(f"whole-frame F0 change from the previous session of the same neuron: median {np.median(f0rows):.1f}%, "
      f"range {f0rows.min():.1f} to {f0rows.max():.1f}%; within 10% in {int((np.abs(f0rows) <= 10).sum())} of {len(f0rows)}")

Not judged: at each neuron's highest field, dF drops from the last 100 ms of the stimulus to 10
to 40 ms after it in 20 of 20 neurons (median drop 3.04 points under both ROIs shown), and the
next session's F0 is within 10% of the one before in 246 of 262 consecutive pairs.

## figs2-ttx-abolishes-aps

> 1 uM tetrodotoxin abolished the APs evoked by unmodulated 250-ms, 2-kHz sine waves.

Per TTX neuron, pairs at the same voltage before and after TTX (O5). Reproduced when no session
after TTX has an AP at a voltage where APs were detected before.

In [ ]:
rows = []
detail = {}
for key, g in M.groupby(KEYS):
    if key[3] != "5sd":  # O4 does not enter this claim
        continue
    for o5 in ("every", "highest"):
        p = sw.pairs(g, "TTX", o5)
        q = p[p["ap_before"] > 0]
        bad = q[q["ap_after"] > 0]
        detail[key[:3] + (o5,)] = p
        rows.append(dict(zip(KEYS[:3], key[:3]), o5=o5, neurons=p["subject"].nunique(), pairs=len(p),
                         pairs_ap_before=len(q), pairs_ap_after=len(bad),
                         aps_before=int(q["ap_before"].sum()), aps_after=int(q["ap_after"].sum()),
                         outcome="none qualify" if len(q) == 0 else ("reproduced" if len(bad) == 0 else "not reproduced")))
r4 = pd.DataFrame(rows)
print(r4.to_string(index=False))
print("\noutcome under the claims file's readings:", overall(r4.assign(sdr="5sd")))
print("\nAP counts before and after per pair, first reading (otsu/all/5pct, every):")
print(detail[("otsu", "all", "5pct", "every")][["subject", "volt", "ap_before", "ap_after"]].to_string(index=False))
print("\npairs with an AP after TTX, per reading:")
for k, p in detail.items():
    bad = p[(p["ap_before"] > 0) & (p["ap_after"] > 0)]
    if len(bad):
        print(k, bad[["subject", "volt", "ap_before", "ap_after"]].values.tolist())

The peaks counted as APs after TTX, with their heights and widths, and the noise of those ROIs.

In [ ]:
after = M[(M["group"] == "TTX") & (M["phase"] == "after") & (M["n_ap"] > 0) & (M["sdr"] == "5sd")]
print(after[["roi", "f0", "ap", "subject", "volt", "n_ap", "ap_peaks", "ap_widths", "pre_sd", "ap_level"]]
      .assign(ap_peaks=lambda d: d["ap_peaks"].apply(lambda v: np.round(v, 1).tolist()),
              ap_widths=lambda d: d["ap_widths"].apply(lambda v: np.round(v, 1).tolist()))
      .round(2).to_string(index=False))
print("\npre-stimulus SD of dF (%) by ROI reading, median over sessions:",
      M[M["sdr"] == "5sd"].groupby("roi")["pre_sd"].median().round(2).to_dict())

**Result: blocked (methods_underspecified).** The claims file's readings give different
outcomes. With the O1(a) ROI, no AP is found after TTX at the highest common field (O5 b), but
over every common field (O5 a) one pair has a single AP after TTX: a 2.8-ms peak of 19.7 to 21.3%
in a 14_20240731 session whose O1(a) ROI is one pixel (pre-stimulus SD 11.8%). With the O1(b)
ROI, left with 3 TTX neurons, the 5% height finds peaks after TTX in 9 of 12 pairs, 2.0 to 3.9 ms
wide, in ROIs whose pre-stimulus SD is about 3 to 12%; the 5-SD height finds none. In the
whole-frame variant no AP is found after TTX in any of 31 pairs under any reading, against 215
to 253 APs before. Over every common field, the O1(a) readings find 158 to 204 APs before and at
most 1 after.

## figs2-ttx-blocks-sustained-depolarization

> Tetrodotoxin also fully blocked the sustained depolarization caused by intense sine waves,
> consistent with the depolarization arising from repeated activation of voltage-gated sodium
> channels by the cycling de- and hyperpolarization of the membrane.

Pairs of a TTX neuron where sustained depolarization (O4) was present before TTX. The criterion
blocks the claim (other) when no TTX neuron has one.

In [ ]:
rows = []
for key, g in M.groupby(KEYS):
    for o5 in ("every", "highest"):
        p = sw.pairs(g, "TTX", o5)
        q = p[p["sd_before"]]
        ok = (~q["sd_after"]) & (q["last100_after"] < q["level_after"])
        rows.append(dict(zip(KEYS, key), o5=o5, pairs=len(p), pairs_sd_before=len(q),
                         neurons_sd_before=q["subject"].nunique(), blocked_after=int(ok.sum()),
                         outcome="none qualify" if len(q) == 0 else ("reproduced" if ok.all() else "not reproduced")))
r5 = pd.DataFrame(rows)
print(r5.to_string(index=False))
print("\noutcome under the claims file's readings:", overall(r5))

**Result: blocked (other).** The criterion blocks the claim when no TTX neuron has sustained
depolarization (O4) before TTX, and none has under any combination of the claims file's
readings. In the whole-frame variant one TTX neuron qualifies under the 5-SD level, and TTX
removes its depolarization.

Variant, not judged: the mean dF over the last 100 ms of the stimulus before and after TTX in
every pair at the same voltage (O5 a), with a two-sided paired t test (F6) over pairs.

In [ ]:
for roi in ("otsu", "otsu_eroded", "frame"):
    for f0r in ("all", "last20"):
        g = M[(M["roi"] == roi) & (M["f0"] == f0r) & (M["ap"] == "5pct") & (M["sdr"] == "5sd")]
        for o5 in ("every", "highest"):
            p = sw.pairs(g, "TTX", o5)
            if len(p) < 2:
                print(roi, f0r, o5, "pairs", len(p))
                continue
            t, pv = stats.ttest_rel(p["last100_before"], p["last100_after"])
            d = p["last100_before"] - p["last100_after"]
            print(f"{roi:11s} {f0r:6s} {o5:7s} pairs {len(p):2d}, neurons {p['subject'].nunique()}: before "
                  f"{p['last100_before'].mean():.2f}%, after {p['last100_after'].mean():.2f}%, difference "
                  f"{d.mean():.2f} (95% CI {d.mean() - stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)):.2f} to "
                  f"{d.mean() + stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)):.2f}), t = {t:.2f}, p = {pv:.2g}")
g = M[(M["roi"] == "frame") & (M["f0"] == "all") & (M["ap"] == "5pct") & (M["sdr"] == "5sd")]
p = sw.pairs(g, "TTX", "every")
print("\nper neuron, whole frame (every pair):")
print(p.groupby("subject")[["last100_before", "last100_after"]].mean().round(2).to_string())
v = sw.pairs(g, "vehicle", "every")
dv = v["last100_before"] - v["last100_after"]
print(f"vehicle pairs for comparison (whole frame): before {v['last100_before'].mean():.2f}%, after "
      f"{v['last100_after'].mean():.2f}%, difference {dv.mean():.2f}, paired t p = {stats.ttest_rel(v['last100_before'], v['last100_after']).pvalue:.2g} ({len(v)} pairs)")
aft = M[(M["roi"] == "frame") & (M["f0"] == "all") & (M["ap"] == "5pct") & (M["sdr"] == "5sd") &
        (M["group"] == "TTX") & (M["phase"] == "after")]
print("whole frame, every TTX session after TTX: mean dF over the last 100 ms, by voltage:")
print(aft.groupby("volt")["last100_mean"].agg(["count", "mean", "min", "max"]).round(2).to_string())

Not judged: in every reading the mean dF over the last 100 ms of the stimulus is lower after TTX
than before at the same voltage (whole frame, every pair: 1.31% before against -0.59% after, 31
pairs, p = 1.6e-07; O1(a): 1.21% against -0.64%, p = 9.5e-07), and after TTX it is below zero
at every voltage up to 10 V and averages -0.25% at 12 V. In vehicle pairs it falls by 0.25 points
(1.80% against 1.55%, p = 0.00054), against 1.91 points in TTX pairs.

## figs2-vehicle-keeps-response

> In vehicle controls, recorded about 5 min after the solution was replaced by the same
> solution without TTX, the firing and sustained depolarization persisted.

Vehicle neurons with a visible solution change (11). Reproduced when every pair with APs before
has APs after, and every pair with sustained depolarization before has it after.

In [ ]:
rows = []
vdetail = {}
for key, g in M.groupby(KEYS):
    for o5 in ("every", "highest"):
        p = sw.pairs(g, "vehicle", o5)
        vdetail[key + (o5,)] = p
        qa = p[p["ap_before"] > 0]
        qs = p[p["sd_before"]]
        lost_ap = int((qa["ap_after"] == 0).sum())
        lost_sd = int((~qs["sd_after"]).sum())
        n_q = len(qa) + len(qs)
        rows.append(dict(zip(KEYS, key), o5=o5, neurons=p["subject"].nunique(), pairs=len(p),
                         pairs_ap_before=len(qa), lost_ap=lost_ap, pairs_sd_before=len(qs), lost_sd=lost_sd,
                         outcome="none qualify" if n_q == 0 else
                         ("reproduced" if lost_ap == 0 and lost_sd == 0 else "not reproduced")))
r6 = pd.DataFrame(rows)
print(r6.to_string(index=False))
print("\noutcome under the claims file's readings:", overall(r6))
print("\npairs that lose their APs, per reading (subject, V, APs before, APs after):")
for k, p in vdetail.items():
    if k[3] != "5sd":
        continue
    bad = p[(p["ap_before"] > 0) & (p["ap_after"] == 0)]
    if len(bad):
        print(k, bad[["subject", "volt", "ap_before", "ap_after"]].values.tolist())

In [ ]:
for key in [("otsu", "all", "5pct", "5sd", "every"), ("frame", "all", "5pct", "5sd", "every")]:
    p = vdetail[key]
    d = p["ap_after"] - p["ap_before"]
    print(f"{key}: AP count before {p['ap_before'].sum()} and after {p['ap_after'].sum()} over {len(p)} pairs; "
          f"fewer after in {int((d < 0).sum())}, more in {int((d > 0).sum())}, equal in {int((d == 0).sum())}; "
          f"paired t p = {stats.ttest_rel(p['ap_before'], p['ap_after']).pvalue:.2g}")
    g = M[(M["roi"] == key[0]) & (M["f0"] == key[1]) & (M["ap"] == key[2]) & (M["sdr"] == key[3]) & (M["group"] == "vehicle")]
    mr = g.groupby(["subject", "phase"])["max_rate_hz"].max().unstack()
    print("  highest maximum firing rate (Hz) per vehicle neuron, before and after:")
    print("  ", mr.round(0).to_dict("index"))

**Result: blocked (methods_underspecified).** The readings disagree. Over every common voltage
(O5 a) some pairs with APs before have none after: 3 or 4 of 50 to 59 under O1(a), 1 of 9 under
O1(b) with the 5% height (with the 5-SD height no O1(b) pair has APs before), and 2 of 59 in the
whole-frame variant, so the claim fails there. At the highest common voltage (O5 b) every pair
keeps its APs under the 5-SD height with O1(a), and 1 of 11 loses them under O1(a) with the 5%
height. No pair has sustained
depolarization (O4) before, so that part of the criterion is empty. Under O1(a) with the 5%
height, vehicle neurons fire 334 APs before and 327 after over 59 pairs (p = 0.77).

## ap-amplitude-duration

> Neurons were selected for APs peaking above 5% dF and lasting about 10 ms; typical APs were
> 11.3% dF high and 8 ms long, excluding the afterhyperpolarization.

Per neuron, the mean peak dF and duration of its APs before the change; the mean over neurons is
compared with 11.3 +/- 0.9% and 8 +/- 0.7 ms. Readings: which APs (all, or the first of each
session) and the duration (width at half height, or rise to the return to F0).

In [ ]:
rows = []
for key, g in M[M["phase"] == "before"].groupby(KEYS[:3] + ["sdr"]):
    if key[3] != "5sd":
        continue
    for which in ("all", "first"):
        per = []
        for subj, h in g.groupby("subject"):
            pk, wd, tf = [], [], []
            for _, r in h.iterrows():
                fr, pkv, wv = r["ap_frames"], r["ap_peaks"], r["ap_widths"]
                if len(fr) == 0:
                    continue
                idx = range(len(fr)) if which == "all" else [0]
                for i in idx:
                    pk.append(pkv[i])
                    wd.append(wv[i])
                    tf.append(sw.duration_to_f0(np.nan_to_num(r["x"], nan=0.0), r["rate"], int(fr[i])))
            if pk:
                per.append(dict(subject=subj, peak=np.mean(pk), fwhm=np.mean(wd), to_f0=np.nanmean(tf),
                                n=len(pk), to_f0_missing=int(np.isnan(tf).sum())))
        per = pd.DataFrame(per)
        for dur in ("fwhm", "to_f0"):
            mp_, md_ = per["peak"].mean(), per[dur].mean()
            sem_p = per["peak"].std(ddof=1) / np.sqrt(len(per))
            sem_d = per[dur].std(ddof=1) / np.sqrt(len(per))
            inside = abs(mp_ - 11.3) <= 0.9 and abs(md_ - 8) <= 0.7
            rows.append(dict(roi=key[0], f0=key[1], ap=key[2], which=which, duration=dur, neurons=len(per),
                             aps=int(per["n"].sum()), peak_mean=round(mp_, 2), peak_sem=round(sem_p, 2),
                             dur_mean=round(md_, 2), dur_sem=round(sem_d, 2),
                             to_f0_missing=int(per["to_f0_missing"].sum()) if dur == "to_f0" else 0,
                             peak_ok=abs(mp_ - 11.3) <= 0.9, dur_ok=abs(md_ - 8) <= 0.7,
                             outcome="within" if inside else "outside"))
r7 = pd.DataFrame(rows)
print(r7.to_string(index=False))
print("\nreadings within both tolerances (claims file's ROIs):",
      int((r7[r7["roi"].isin(CLAIMS_ROI)]["outcome"] == "within").sum()), "of", int(r7["roi"].isin(CLAIMS_ROI).sum()))
nb = M[(M["phase"] == "before") & (M["sdr"] == "5sd") & (M["ap"] == "5pct")].groupby(["roi", "f0"])
print("neurons with APs above 5% dF before the change:", nb.apply(lambda g: g.loc[g["n_ap"] > 0, "subject"].nunique()).to_dict(),
      "of", B["subject"].nunique())

**Result: blocked (methods_underspecified).** One reading (O1 a, F0 over all frames, 5-SD
height, first AP of each session, duration to F0) gives means within the paper's SEM (10.76% and
7.62 ms over 18 neurons); the other 31 combinations of the claims file's readings do not. With
the O1(a) ROI, mean peaks are 9.29 to 10.82% and widths at half height 3.56 to 4.06 ms; with
the O1(b) ROI (7 neurons, 1 with the 5-SD height) peaks are 16.3 to 18.9%. In the whole-frame
variant peaks are 7.67 to 9.54% and widths 3.79 to 5.07 ms. The width at half height is about
half the paper's 8 ms in every reading.

## fig3-not-heating (calculated, no data)

> Excitation by pure sine waves was not caused by heating; the adiabatic temperature rise was
> 2.5 C after 250 ms at 63 V/cm in NaCl solution, at most 1.24 C for the same stimulus in the
> low-conductivity solution, and 4.1 C at 81 V/cm in NaCl.

The rise is sigma E^2 t / (rho c), with the heat capacity of water, E as (a) the peak and (b)
the RMS field (criterion). The conductivities are the Methods' 16.4 and 8.4 mS/cm.

In [ ]:
RHO_C = 4.18e6  # J per m3 per K, water (criterion)
cases = [("63 V/cm, NaCl", 63, 1.64, 2.5), ("63 V/cm, NaGlu", 63, 0.84, 1.24), ("81 V/cm, NaCl", 81, 1.64, 4.1)]
res = []
for name, e_vcm, sigma, paper in cases:
    for reading, fac in (("peak", 1.0), ("rms", 1 / np.sqrt(2))):
        e = e_vcm * 100 * fac
        dt = sigma * e ** 2 * 0.25 / RHO_C
        res.append(dict(case=name, reading=reading, rise_C=round(dt, 3), paper=paper,
                        within_10pct=abs(dt - paper) <= 0.1 * paper))
res = pd.DataFrame(res)
print(res.to_string(index=False))
r81 = (81 / 63) ** 2
rglu = 0.84 / 1.64
print(f"ratio 81/63 V/cm: formula {r81:.3f}, paper {4.1 / 2.5:.3f} (difference {100 * (4.1 / 2.5 / r81 - 1):.1f}%)")
print(f"ratio NaGlu/NaCl: formula {rglu:.3f}, paper {1.24 / 2.5:.3f} (difference {100 * (1.24 / 2.5 / rglu - 1):.1f}%)")
print(f"heat capacity that would give 2.5 C with the peak field: {1.64 * 6300 ** 2 * 0.25 / 2.5 / 1e6:.2f} MJ/m3/K; "
      f"with the RMS field: {1.64 * 6300 ** 2 / 2 * 0.25 / 2.5 / 1e6:.2f} MJ/m3/K")

**Result: blocked (methods_underspecified).** Neither reading gives the paper's temperatures
(peak field 3.89, 1.99 and 6.44 C; RMS field 1.95, 1.00 and 3.22 C, against 2.5, 1.24 and
4.1 C), while the paper's ratios follow the formula (81/63 V/cm within 0.8%, NaGlu/NaCl within
3.2%), which the criterion rules blocked. Getting 2.5 C would need a heat capacity of 6.51
MJ/m3/K with the peak field or 3.25 with the RMS field, against 4.18 for water.

## Summary

Outcomes of all claims of the claims file (this notebook, `reproduce_fig3c_001030.ipynb` and
`reproduce_fig1_fig2_001032.ipynb`):

| Claim | Data | Outcome |
|---|---|---|
| figs2-neurons-tested | 001271 | reproduced |
| figs2-before-firing-then-depolarization | 001271 | partially reproduced |
| figs2-depolarization-reversible | 001271 | blocked: other (no session with sustained depolarization under O4) |
| figs2-ttx-abolishes-aps | 001271 | blocked: methods underspecified (the ROI readings disagree) |
| figs2-ttx-blocks-sustained-depolarization | 001271 | blocked: other (no TTX neuron with sustained depolarization under O4) |
| figs2-vehicle-keeps-response | 001271 | blocked: methods underspecified (the pair readings disagree) |
| ap-amplitude-duration | 001271 | blocked: methods underspecified (1 of 32 readings within the paper's values) |
| fig3-not-heating | calculated | blocked: methods underspecified (heat capacity and field definition unknown) |
| fig3c-sustained-firing-2s | 001030 | partially reproduced |
| fig1-pure-sine-excites-all | 001032, 001033, 001035 | not reproduced |
| fig1-20khz-needs-twice-the-field | 001032, 001033, 001035 | not reproduced |
| fig2d-modulated-thresholds-higher | 001032, 001033, 001035 | not reproduced |
| fig2-modulated-locks-to-beats | 001032, 001033, 001035 | partially reproduced (assumption) |
| fig2c-pure-more-aps | 001032, 001033, 001035 | reproduced (assumption) |
| fig3b-low-conductivity-halves-threshold | none found | blocked: metadata insufficient (no dandiset found to hold it) |
| figs1-di8-same-as-fluovolt | 001272 | not reproduced (3 neurons where the paper has 8) |
| fig4-pooled-time-constants, fig4-orientation-and-poles | 001273 | blocked: other (not attempted in this run) |
| fig6-sine-less-than-pulse | 001274 | blocked: other (not attempted in this run) |
| threshold-depolarization-estimate | 001032 to 001035, 001274 | blocked: other (depends on fig6) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper measured F over a soma outline drawn in
ImageJ or MetaMorph on 64 x 64 images and identified APs by eye. The files hold 16 x 16 crops and
no outline, so the claims file's Otsu ROIs (O1) and AP rule (O3) are used; the Otsu component is
often a few pixels of bright membrane and the 2-pixel erosion of O1(b) is empty in 210 of 287
sessions, which drives most of the disagreements between readings. The claims file's measure of
sustained depolarization (O4) asks for a plateau above 5 pre-stimulus SD or half the AP height
with no AP in the last 100 ms; the plateaus in these data are lower (median 2.6% at the highest
field). The solution change of vehicle neurons is placed at the last gap over 3 minutes, which the
files do not mark. The first and last frame of every file are left out.

What was found about the dataset. It holds the Figure S2 experiment: 6 TTX and 11 vehicle
neurons with a visible solution change, 3 more vehicle-labelled neurons without one, and 5 sham
sessions; no file is deposited twice. Its voltages give the fields of Figure S2 at 18 V/cm per V,
not the 36 V/cm per V the files state. The TTX descriptions say TTX "caused sustained
depolarization", where the paper and the dandiset description say it prevented it; the data side
with the paper. Sessions before the change carry the same "vehicle control" label as vehicle
sessions after it. The files hold no stimulus record, no ROI and frames of 16 x 16 pixels.